# Minimal THC example for vibrational Hamiltonian

The purpose of this notebook is to demonstrate the usage of the `vib_frags` package for performing (BLISS-)THC factorization of two-mode coupling vibrational Hamiltonians, as well as obtaining the associated costs for QPE. Let $g_{i,p,q,j,r,s}$ be the symmetrized two-mode-coupling tensor appearing in Eq. (11) of the associated paper. This symmetry allows for the exchange of modal indices within a mode, or composite mode-modal indices, without changing the tensor element. The THC factorization can be given as
\begin{equation*}
  g_{i,p,q,j,r,s} \approx \sum_{\mu_i = 0}^{R_i-1} \sum_{\nu_j = 0}^{R_j-1} \zeta^{(i,j)}_{\mu_i, \nu_j} \xi^{(i)}_{p,\mu_i} \xi^{(i)}_{q,\mu_i} \xi^{(j)}_{r,\nu_j} \xi^{(j)}_{s,\nu_j}
\end{equation*}
where $\zeta^{(i,j)}_{\mu_i, \nu_j} = \zeta^{(j,i)}_{\nu_j, \mu_i}$ and $\sum_p |\xi^{(i)}_{p,\mu_i}|^2 = 1$ for all $i,j,\mu_i,$ and $\nu_j$.

We find such a factorization by a non-linear optimization using $\zeta$ and $\xi$ tensors as parameters to minimize the error in the above approximation. The error is calculated as the Frobenius norm of the difference between the original tensor on the left side and the reconstructed tensor on the right side. However, to obtain an efficient factorization, it is best to start with a good initial guess. Below, we start by obtaining the initial guess, followed by performing the full THC optimization.

# CP3 initialization

A good initial guess using CP3 decomposition can be obtianed as follows. Start by performing an eigendecomposition on the matrix $G$, which is obtained by reshaping the tensor $g$:

$$ g_{i,p,q,j,r,s} \equiv G_{(i,p,q),(j,r,s)} = \sum_\alpha \sigma_\alpha \mathcal{G}_{(i,p,q),\alpha} \mathcal{G}_{(j,r,s),\alpha}$$

This decouples the mode indices, allowing us to perform a CANDECOMP/PARAFAC decomposition of rank 3 (CP3) factorization for each mode as
$$
  \mathcal G_{(i,p,q),\alpha} \approx \sum_{\mu_i = 0}^{R_i-1}\xi^{(i)}_{p,\mu_i}\xi^{(i)}_{q,\mu_i}\bar D^{(i)}_{\alpha, \mu_i}.
$$


Substituting this back in to the eigendecomposition of $G$, we get
$$
g_{i,p,q,j,r,s}^{\mathrm{THC}} = \sum_{\mu_i = 0}^{R_i-1} \sum_{\nu_j = 0}^{R_j-1}
\left(
\sum_\alpha \sigma_\alpha  \bar D^{(i)}_{\alpha, \mu_i} \bar D^{(j)}_{\alpha, \nu_j}\right) \xi^{(i)}_{p,\mu_i}\xi^{(i)}_{q,\mu_i} \xi^{(j)}_{r,\nu_j}\xi^{(j)}_{s,\nu_j},
$$
where we identify $\zeta^{(i,j)}_{\mu_i,\nu_j} = \sum_\alpha \sigma_\alpha  \bar D^{(i)}_{\alpha, \mu_i} \bar D^{(j)}_{\alpha, \nu_j}$.
Below we show how to obtain the $\xi$ and $\zeta$ tensors using CP3.

## Importing modules

In [ ]:
import numpy as np
import h5py

from vib_frags import cp3_utils as cpu
from vib_frags import hthc_decomp as hthc
from vib_frags import tensor_utils as tu
from vib_frags import cost_utils as cu

## Importing Hamiltonian

Hamiltonians used for Trotter estimates are smaller and are stored in a different convention from the PAH Hamiltonians. That distinction is made in how the loaded tensors are transposed.

In [ ]:
mol_name = 'H2S'                                                     #Can be 'pah1', 'pah2', ..., 'pah7', or 'H2S', 'CO2', 'CH2O', 'CH4' 
au_to_cm1 = 219474.6                                                  #Stored tensors are in atomic units
mc_level = 2                                                          #Mode-coupling level
nmodals = 4                                                           #Number of modals
Ham_source = 'Trotter'                                                    #Can be 'Trotter' or 'PAH'. In the former case, the Hamiltonians considered are much smaller than the PAH Hamiltonians

#Loading the Hamiltonian
if Ham_source == 'Trotter':
  datafile = h5py.File(rf'Hamiltonians/Trotter_Hamiltonians/{mol_name}_loc.hdf5', 'r')
elif Ham_source == 'PAH':
  datafile = h5py.File(rf'Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/{mol_name}_VSCF_{nmodals}M_{mc_level}MC.hdf5', 'r')

#Extracting Hamiltonian terms at different mode-coupling levels
#One-mode-coupling term
H1_full = np.array(datafile['H1'][()]) * au_to_cm1     
H1 = H1_full[:, :nmodals, :nmodals]

#Two-mode-coupling term
H2_full = np.array(datafile['H2'][()]) * au_to_cm1      
if Ham_source == 'Trotter':
  H2_nonsym = np.transpose(H2_full, (0,2,4,1,3,5))                          #Permuting axes to match the convention in the paper
elif Ham_source == 'PAH':
  H2_nonsym = np.transpose(H2_full, (3,4,5,0,1,2))
H2_nonsym = H2_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals]       #Non symmetrized tensor
H2 = tu.symmetrize_tbt(H2_nonsym)

#Three-mode-coupling term
if Ham_source == 'PAH':
  try:
    H3_full = np.array(datafile['H3'][()]) * au_to_cm1
    H3_nonsym = np.transpose(H3_full, (6,7,8,3,4,5,0,1,2))
    H3_nonsym = H3_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals, :, :nmodals, :nmodals]
    H3 = tu.symmetrize_trbt(H3_nonsym)
  except KeyError:
    print ("3 mode term is missing. Setting 3 mode tensor to None.")
    H3_full, H3_nonsym, H3 = None, None, None
elif Ham_source == 'Trotter':
  H3_full, H3_nonsym, H3 = None, None, None
  mc_level = 2

nmodes = H1.shape[0]
print ('Number of modes:', nmodes)
print ('Number of modals:', nmodals)

## Preparing for CP3

In [ ]:
G_mat = H2.reshape(nmodes*nmodals**2, nmodes*nmodals**2)                    #The G matrix explained above
G_val, G_vec = np.linalg.eigh(G_mat)
G_val_mask = np.where(np.abs(G_val)>0)[0]                                   #Filtering small eigenvalues if needed
G_val = G_val[G_val_mask]
G_vec = G_vec[:, G_val_mask]
print ("Number of retained eigenvectors:", len(G_val))

G_val_sign_mat = np.diag(np.sign(G_val))                                    #Signs of the eignvalues will be separated before performing CP3
G_vec_scld = G_vec @ np.diag(np.sqrt(np.abs(G_val)))                        #But the magnitudes will be absorbed into the eigenvectors
G_vec_ten = G_vec_scld.reshape(nmodes, nmodals, nmodals, len(G_val))        #Reshaping the scaled G eigenvectors to perform CP3 on

## Performing CP3

For many systems studied in this work, CP3 initialization often works exceptionally well with THC providing only about a factor of two reduction in the final error compared to CP3. However, this could change as the number of modals becomes large. In this example, we consider a constant THC rank across modes. But in practice, one needs to first run a CP3 optimization with a low THC rank (e.g. ~ nmodals + 1) and then increase it for cases where the residual error is large.

In [ ]:
#Containers to collect outputs of CP3 function for each mode. thetas are the angles used to parameterized the \xi vectors.
#This is possible since \xi^{(i)} tensors have L2 normalized columns.
thetas_dict = {}
Cs_dict = {}
cp3_errs = {}
Nthc_list = {0: 4, 1: 4, 2: 3}                                                  #Stores the THC rank R_i for mode i. For large systems, usually R_i = nmodals + 1 is a good start.

rho = 1e-5                                                      #A regularization coefficient to limit the growth of |D_{\alpha,\mu}| as this later contributes to LCU 1 norm via zeta

# CP3 for each mode
for i in range (nmodes):
    print ("mode : ", i)
    Ci = G_vec_ten[i]
    theta_fit, C_fit = cpu.fit_cp_symmetric(Ci, rank=Nthc_list[i], n_steps=3000, lr=3e-3, rho=rho, tol = 1e-6, check_convergence = False, window_size=600, initial_guess = None)
    thetas_dict[i], Cs_dict[i] = theta_fit, C_fit
    Nthc_list[i] = thetas_dict[i].shape[1]

    A_fit = cpu.angles_to_unit_vectors(theta_fit)
    Z_hat = cpu.reconstruct_tensor(A_fit, C_fit)
    cp3_err_i = 0.5 * np.sum((Z_hat - Ci) ** 2)
    cp3_errs[i] = float(cp3_err_i)

#Repacking CP3 outputs for THC input
zetas_dict = {}
Nthc_list = []
for i in range (nmodes):
    for j in range (i):
        #The factor of 2 below is used since Cs are obtained for symmetrized Hamiltonian but we want to use zeta for unsymmetrized Hamiltonian
        zetas_dict[(i,j)] = 2*(Cs_dict[i].T @ G_val_sign_mat @ Cs_dict[j])      
    Nthc_list.append(thetas_dict[i].shape[1])
hcp3_params = {'theta': thetas_dict, 'zeta': zetas_dict, 'Nthc': Nthc_list, 'Cs': Cs_dict}


cp3_eps_ten = float(hthc.tbt_error(thetas_dict, zetas_dict, H2_nonsym))
print ("\nAbsolute tensor error due to CP3:", cp3_eps_ten, "cm-1")
Ravg = round(np.mean(Nthc_list),2)
print ("Average THC rank: ", Ravg)

## Performing THC

The regularization parameter here controls the 1-norm contribution to the total loss function. Often times, it was observed that increasing this parameter helps in pushing the optimizer out of local minima.

In [ ]:
#To perform THC without warm start, set initial_guess = None. Note that this allows for a higher learning rate than when you have a very good warm start.
#Regularize parameter allows for minimizing the one-norm along with the tensor approximation error
hthc_params, one_norm = hthc.get_vib_hthc(H1, H2_nonsym, trbt=None, maxiter = 100000, Nthc = hcp3_params['Nthc'], learning_rate = 1e-3, regularize=5e-3, initial_guess = hcp3_params, chunk_size=500)

# Note that hthc_params.keys() = dict_keys(['theta', 'zeta', 'Nthc'])

#Hamiltonian can be reconstructed as,
H2_nonsym_THC, _ = hthc.unfold_vib_hthc(hthc_params['theta'], hthc_params['zeta'])

#Error in THC approximation
thc_eps_ten = np.sqrt(np.sum((H2_nonsym - H2_nonsym_THC)**2))
print ("THC error :", thc_eps_ten, "cm-1")

## Performing BLISS-THC
Here, we modify only the one-body term through symmetry shift and keep the two body term intact.

In [ ]:
H1_THC_modified = H1 + np.einsum('ipqjrr -> ipq', H2)

# In only the one-mode-coupling operator is being modified, optimization is not required as the optimal BLISS parameters can be obtained exactly
bliss_thc_params = {}
bliss_thc_params['A'] = np.median(np.linalg.eigvalsh(H1_THC_modified), axis=1)

## Calculating the 1-norm and the QPE cost

If bliss is set to True and the above cell has was run successfully, then the cost evaluated here corresponds to BLISS-THC. The gate cost correspond to the Toffoli count. The qubit cost corresponds to the number of logical qubits utilized by the algorithm. By default, the number of bits of precision used to specify the Hamiltonian coefficients and the rotation angles is set to 20.

In [ ]:
eps_QPE = 10                                       # Target error in energy
bliss = True          

if bliss == False:
  bliss_thc_params = None

nthc = hthc_params['Nthc']
zeta = hthc_params['zeta']
theta = hthc_params['theta']
H2_hthc_nonsym,_ = hthc.unfold_vib_hthc(theta, zeta)
rel_error = np.linalg.norm(H2_hthc_nonsym - H2_nonsym)/np.linalg.norm(H2_nonsym)

# Calculating the (BLISS-)THC LCU 1-norm
lmbda = float(np.sum(hthc.hthc_vib_one_norm(obt=H1, tbt=H2_nonsym, trbt=None, zeta=zeta, gamma=None, obt_is_none=False, trbt_is_None=True, Rthc = max(nthc), bliss_params=bliss_thc_params)))

# Can be "clean", "dirty", "QROM", or "None". The last two essentailly implement the same thing
QROAM_type = "dirty"
select_copy = False

if QROAM_type == "dirty":
  dirty, clean = True, False
elif QROAM_type == "clean":
  dirty, clean = False, 500                                             #500 here is the number of additional clean ancillas available if QROAM_type = "clean"
elif (QROAM_type == "None") or (QROAM_type == "QROM"):
  dirty, clean = False, False

C_THC, N_THC = cu.get_THC_QPE_cost(lmbda, eps_QPE, nmodes, nmodals, hthc_params['Nthc'], mc_level, br = 20, aleph = 20, beth = 20, dirty = dirty, clean = clean , select_copy = select_copy)

print ("\nMolecule: ", mol_name)
print ("Modals per mode: ", nmodals)
print ("THC rank: ", nthc)
print ("Avg THC rank: ", np.mean(np.array(nthc)))
print ("\nRelative error in tensor approximation = ", np.round(rel_error,4),"cm-1")
print (f"Absolute error in tensor approximation: {np.round(thc_eps_ten,4)} cm-1")
print (f"Target accuracy in energy: {eps_QPE} cm-1")